### Data Wrangling

In [22]:
import pandas as pd
import statsmodels.api as sm

In [23]:
data = pd.read_csv("dataSetMain.csv") # contains 40 entries of high-quality data
data.head()

,Project_ID,Repository_URL,Language,Framework,Team_Size,Lines_of_Code,Commit_Count,Contributors,Bug_Count,Security_Vulnerabilities,...,Test_Automation_Percent,Dependency_Count,Outdated_Dependencies_Count,Container_Usage,Cloud_Platform,Monitoring_Tools_Count,Security_Scan_Score,Performance_Benchmark_MS,Memory_Usage_MB_Avg,CPU_Usage_Percent_Avg
0,SE_001,github.com/techcorp/webapp-frontend,TypeScript,React,8,125000,2340,12,89,3,...,95,67,8,Docker,AWS,5,9.2,245,512,12.5
1,SE_002,github.com/fintech/payment-api,Python,FastAPI,15,89000,3450,18,45,1,...,98,123,15,Kubernetes,GCP,8,9.5,123,256,8.9
2,SE_003,bitbucket.org/enterprise/crm-system,Java,Spring_Boot,25,234000,5670,28,156,8,...,85,234,45,Docker,Azure,6,7.8,567,1024,18.7
3,SE_004,github.com/startup/mobile-app,Dart,Flutter,6,67000,1890,8,34,2,...,92,89,12,NaN,Firebase,3,8.9,189,128,6.5
4,SE_005,gitlab.com/opensource/data-pipeline,Python,Apache_Airflow,12,156000,4230,22,78,5,...,96,187,23,Docker,AWS,7,9.1,345,768,15.2


### One Hot Encoding

In [24]:
lang_dummies = pd.get_dummies(data['Language'], prefix='Lang', drop_first=True)
X = data[['Team_Size', 'Lines_of_Code', 'Commit_Count', 'Dependency_Count',
              'Outdated_Dependencies_Count']]

# Add language dummy columns


# Multiple Regression Model

## Setup


### One Hot Encoding

Allows our 'languages' column to used as it contains categorical data such as __python__ or __C++__

In [25]:

lang_dummies = pd.get_dummies(
    data['Language'],
    prefix='Lang',
    drop_first=True,
    dtype=float
)




### Predictor Variable Setup

In [26]:
X = data[['Team_Size', 'Lines_of_Code', 'Commit_Count',
              'Dependency_Count', 'Outdated_Dependencies_Count']]
X = pd.concat([X, lang_dummies], axis=1)
X = X.apply(pd.to_numeric, errors='coerce')

X = sm.add_constant(X)

### Response Variable Setup

In [27]:

y_bugs = pd.to_numeric(data['Bug_Count'], errors='coerce')
y_sec = pd.to_numeric(data['Security_Vulnerabilities'], errors='coerce')

# Data Modeling

## Simple Multiple Regression

### Bug Count 

In [28]:
model_bugs = sm.OLS(y_bugs, X).fit()
print(model_bugs.summary())

                            OLS Regression Results                            
Dep. Variable:              Bug_Count   R-squared:                       0.990
Model:                            OLS   Adj. R-squared:                  0.975
Method:                 Least Squares   F-statistic:                     65.08
Date:                Mon, 01 Dec 2025   Prob (F-statistic):           3.10e-11
Time:                        12:53:09   Log-Likelihood:                -129.31
No. Observations:                  40   AIC:                             308.6
Df Residuals:                      15   BIC:                             350.8
Df Model:                          24                                         
Covariance Type:            nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
const             

### Security Vulnerabilities

In [29]:
model_sec = sm.OLS(y_sec, X).fit()
print(model_sec.summary())

                               OLS Regression Results                               
Dep. Variable:     Security_Vulnerabilities   R-squared:                       0.979
Model:                                  OLS   Adj. R-squared:                  0.946
Method:                       Least Squares   F-statistic:                     29.23
Date:                      Mon, 01 Dec 2025   Prob (F-statistic):           1.03e-08
Time:                              12:53:09   Log-Likelihood:                -43.414
No. Observations:                        40   AIC:                             136.8
Df Residuals:                            15   BIC:                             179.1
Df Model:                                24                                         
Covariance Type:                  nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------

## Robust Linear Regression Model 

uses Iteratively Reweighted Least Squares to get a more accurate p value

### Bug Count

In [30]:
rlm_model = sm.RLM(y_bugs, X, M=sm.robust.norms.HuberT())
results_rlm = rlm_model.fit()
print(results_rlm.summary())

                    Robust linear Model Regression Results                    
Dep. Variable:              Bug_Count   No. Observations:                   40
Model:                            RLM   Df Residuals:                       15
Method:                          IRLS   Df Model:                           24
Norm:                          HuberT                                         
Scale Est.:                       mad                                         
Cov Type:                          H1                                         
Date:                Mon, 01 Dec 2025                                         
Time:                        12:53:09                                         
No. Iterations:                    50                                         
                                  coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------
const             

### Security Vulnerabilities

In [31]:
rlm_model2 = sm.RLM(y_sec, X, M=sm.robust.norms.HuberT())
results_rlm2 = rlm_model2.fit()
print(results_rlm2.summary())

                       Robust linear Model Regression Results                       
Dep. Variable:     Security_Vulnerabilities   No. Observations:                   40
Model:                                  RLM   Df Residuals:                       15
Method:                                IRLS   Df Model:                           24
Norm:                                HuberT                                         
Scale Est.:                             mad                                         
Cov Type:                                H1                                         
Date:                      Mon, 01 Dec 2025                                         
Time:                              12:53:09                                         
No. Iterations:                          50                                         
                                  coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------